#### ======================================================

#### XRD-CT Phase Map and Radial Profile Analysis

#### ======================================================

This notebook provides a workflow for processing and visualizing spatially resolved XRD-CT phase maps from catalyst pellets. It is designed for datasets where phase-specific reconstructed maps are available for different catalyst states or reaction times.

The workflow includes:

1. Loading XRD-CT reconstructed phase maps for multiple samples and phases.
2. Organizing the data by sample state and crystalline phase.
3. Defining custom colormaps and plotting publication-style multi-panel phase maps.
4. Applying optional phase-specific thresholds to remove low-intensity background.
5. Generating pellet masks from a selected reference phase, such as e.g.:`Nb2O5 orthorhombic`.
6. Cleaning the pellet mask by retaining the largest connected component, filling internal holes/cracks, and applying mild binary closing.
7. Calculating distance-to-surface radial coordinates:
   - `r/R = 0` for the pellet core
   - `r/R = 1` for the outer pellet region
8. Computing radial profiles for selected phases across different catalyst states.
9. Plotting multi-panel radial profiles with consistent styling and sample ordering.
10. Creating diagnostic plots showing:
    - Phase maps with iso-contours
    - Radial binning histograms
    - Distance-to-surface radial profiles
11. Exporting publication-quality figures to disk.

The notebook is intended as a reproducible analysis pipeline for evaluating spatial phase gradients in XRD-CT datasets of catalyst pellets, especially when irregular pellet shapes make simple center-based radial profiles unreliable.

The notebook is designed to accommodate beamstop-processed XRD-CT phase maps, provided that the reconstructed phase maps are exported in a compatible format within the .h5 file.

This script was developed using an Nb2O_5-based catalyst pellet case study as reference. Therefore, the default variables, phase dictionaries, mask definitions, and plotting settings follow this example dataset, but they can be modified and adapted to other beamstop-processed XRD-CT phase maps.

## Libraries

In [ ]:
import os
import math
import re
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import distance_transform_edt, binary_fill_holes, binary_closing, label

## Tools 

In [ ]:
# ============================================================
# --- Helper functions --------------------------------------
# ============================================================
# Automatic chemical formula to Unicode subscripts

subscript_map = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")

def chem_unicode(name):
    return ''.join(c.translate(subscript_map) if c.isdigit() else c for c in name)


# ============================================================
# --- Input samples ---
# ============================================================

base_dir = "C:/Users/Desktop/XRD_data"

samples = {
    "Calcined": "sample_name_1",
    "Reduced": "sample_name_2",
    "25 h": "sample_name_3"
}

dataset_prefix = "wt_" # chooses datasets with this suffix inside the h5 file
dataset_suffix = "_error"   # ignore datasets with this suffix inside the h5 file

# ============================================================
# --- Phase renaming and color definitions ---
# ============================================================

# e.g. wt_ is the dataset_ prefix that is inside of the .h5 files.
# To plot lattice parameters and other datasets, one must change this suffix.

#------------------------------------------------------------------------
# Optional naming correction for the phases within the h5 file
#------------------------------------------------------------------------
# Notice that one phase can have more than one simple name, therefore,
# correcting it is great for capturing the phases from different samples
# and later labeling them accordingly
#------------------------------------------------------------------------

name_map = {
    dataset_prefix + 'C': 'Carbon',    
    dataset_prefix + 'CLoad': 'Carbon',
    dataset_prefix + 'Fe2O3': 'Fe2O3',
    dataset_prefix + 'FeNb2O6': 'FeNb2O6',
    dataset_prefix + 'Nb2O5ortho': 'Nb2O5 orthorhombic',
    dataset_prefix + 'Nb2O5orth': 'Nb2O5 orthorhombic',
    dataset_prefix + 'Nb2O5tet': 'Nb2O5 tetragonal',
    dataset_prefix + 'NbO076': 'NbO',
    dataset_prefix + 'Ni': 'Ni',
    dataset_prefix + 'RuNew': 'Ru',
    dataset_prefix + 'Ru': 'Ru',
    dataset_prefix + 'RuO2': 'RuO2',
    dataset_prefix + 'nno': 'NiNb2O6',
    dataset_prefix + 'NiO': 'NiO'
    
}


#---------------------------------------------------------------------------------
# Color dictionary for each phase: can contain single colors or lists of 2+ colors
#---------------------------------------------------------------------------------

target_colors = {
    'Carbon': ['#000000', '#89cff0', '#FF00FF',],
    'Fe2O3': '#fc8432',
    'FeNb2O6': '#F78336',
    'Nb2O5 orthorhombic': '#FFFFFF',
    'Nb2O5 tetragonal': ['#000000','#ff7f24','#ffef00', '#fff8e7'],
    'NbO': ['#000000','#6A0DAD','#FF69B4', '#FFA07A'],
    'NiNb2O6': 'cyan',
    'Ni': '#63B8FF',
    'Ru': '#800080',
    'RuO2': '#FC5A80',
    'NiO': '#00FF00',
}


#---------------------------------------------------------------------------------
# Creating the color maps
#---------------------------------------------------------------------------------

def create_black_colormap(colors): # fancy color maps **
    """
    Create a LinearSegmentedColormap from one or more colors.
    Accepts hex strings, named colors, or lists thereof.
    """
    # Ensure it's a list
    if isinstance(colors, str):
        colors = ["black", colors]

    # Convert all colors to RGB tuples (safe for matplotlib)
    rgb_colors = [to_rgb(c) for c in colors]

    return LinearSegmentedColormap.from_list("custom_gradient", rgb_colors)


# Build dictionary of colormaps for all phases
phase_colormaps = {}
for phase, color_value in target_colors.items():
    try:
        phase_colormaps[phase] = create_black_colormap(color_value)
    except ValueError:
        print(f"Warning: invalid color for {phase}, using black→white fallback")
        phase_colormaps[phase] = create_black_colormap(["black", "white"])
        
    

def get_phase_cmap(phase_name):
    """
    Return a colormap for the given phase.
    Special cases for e.g.:Ru and RuO2, otherwise use black->color mapping.
    """
    if phase_name == "Ru":
        return plt.cm.inferno
    elif phase_name == "RuO2":
        return plt.cm.magma
    elif phase_name in target_colors:
        # Use black->color gradient
        return create_black_colormap(target_colors[phase_name])
    else:
        return plt.cm.gray


# ==================================================================
# --- Manual colorbar vmin/vmax for phases (None = automatic) ---
# ==================================================================
# Useful for regulating how the samples look on the phase plot based
# on the values from wt.%
#-------------------------------------------------------------------

manual_vlimits = {
    'Carbon': (None, 40),
    'Fe2O3': (None, 3),
    'Nb2O5 orthorhombic': (None, 100),
    'Nb2O5 tetragonal': (None, 20),
    'NbO': (None, 0.10),
    'Ru': (None, 10),
    'RuO2': (None, 15)
}




## Cropping lines in the data

In [ ]:
# ============================================================
# --- Helper functions for cropping -------------------------
# ============================================================

def get_cropped_lines_im(im, row_start=50, row_end=350, threshold=0.1):
    """
    Find all rows to crop in the middle region of a single image.
    Returns a sorted array of row indices to remove.
    """
    middle = im[row_start:row_end, :]
    keep_rows = np.any(middle >= threshold, axis=1)
    cropped_rows = np.where(~keep_rows)[0] + row_start
    return np.unique(cropped_rows).astype(int)


def crop_lines(im, cropped_rows):
    """
    Crop the specified rows from a single 2D image.
    """
    mask = np.ones(im.shape[0], dtype=bool)
    mask[cropped_rows] = False
    return im[mask, :]


def pad_top_bottom(im, n_top=0, n_bottom=0):
    """
    Pad a 2D image with zeros on top and bottom.
    """
    return np.pad(im, ((n_top, n_bottom), (0, 0)), mode='constant', constant_values=0)


# ============================================================
# --- Collect all datasets (manual crop + global padding) ---
# ============================================================

all_data = {}
all_phases = set()

# Manual removal of rows per sample --- "very nice" for line artifacts
manual_rows = {
    "Calcined": [90,206,240,243,244],
    "Reduced": [13], 
    "25 h": [168]
}

# Step 1: Crop + pad within each sample
sample_stacks = {}  # store temporary cropped stacks per sample

for label, folder in samples.items():
    sample_dir = os.path.join(base_dir, folder)
    h5_files = [f for f in os.listdir(sample_dir) if f.endswith(".h5")]
    if not h5_files:
        print(f"No .h5 files found in {sample_dir}")
        continue

    sample_data = {}
    imgs = []
    phase_names = []

    for f in h5_files:
        path = os.path.join(sample_dir, f)
        with h5py.File(path, "r") as h5f:
            datasets = [d for d in h5f.keys() if d.startswith(dataset_prefix) and not d.endswith(dataset_suffix)]
            datasets.sort()
            for d in datasets:
                phase_name = name_map.get(d.split('#')[0], d)
                img = h5f[d][:]
                # Automatic + manual cropping
                auto_cropped_rows = get_cropped_lines_im(img, row_start=50, row_end=350, threshold=0.1)
                all_cropped_rows = np.unique(np.concatenate([auto_cropped_rows, manual_rows.get(label, [])]))
                img_cropped = crop_lines(img, all_cropped_rows)

                imgs.append(img_cropped)
                phase_names.append(phase_name)
                all_phases.add(phase_name)

    # Pad within this sample to make all images equal height
    max_rows_sample = max(im.shape[0] for im in imgs)
    for i, im in enumerate(imgs):
        n_pad = max_rows_sample - im.shape[0]
        imgs[i] = pad_top_bottom(im, n_top=n_pad)  # pad on top

    # Store in temporary sample stack
    sample_stacks[label] = dict(zip(phase_names, imgs))

# Step 2: Find global max height across all samples
global_max_rows = max(im.shape[0] for stack in sample_stacks.values() for im in stack.values())

# Step 3: Pad all samples to global max height
for label, stack in sample_stacks.items():
    sample_data = {}
    for phase_name, im in stack.items():
        n_pad = global_max_rows - im.shape[0]
        sample_data[phase_name] = pad_top_bottom(im, n_top=n_pad)
    all_data[label] = sample_data

    
# ========================================================================
# --- Define the desired plotting order for phases (top → bottom rows) ---
# ========================================================================

# If the name is removed the phase is also not plotted *

phase_plot_order = [
    'RuO2',
    'Ru',
    'Nb2O5 orthorhombic',
    'Nb2O5 tetragonal',
    'NbO',
    'Carbon'
]

# Keep only phases that exist in your data
phases = [p for p in phase_plot_order if p in all_phases]

# ====================================================================
# --- Compute min/max per phase across all samples (for colorbars) ---
# ====================================================================

phase_vmin_vmax = {}

for phase in phases:
    values = []
    for sample_data in all_data.values():
        if phase in sample_data:
            values.append(sample_data[phase])
    if values:
        stacked = np.stack(values)
        nonzero = stacked[stacked > 0]

        # Determine vmin
        if phase in manual_vlimits and manual_vlimits[phase][0] is not None:
            vmin = manual_vlimits[phase][0]
        else:
            # automatic: use smallest nonzero
            vmin = nonzero.min() if nonzero.size > 0 else 1e-6

        # Determine vmax
        if phase in manual_vlimits and manual_vlimits[phase][1] is not None:
            vmax = manual_vlimits[phase][1]
        else:
            vmax = stacked.max()

        phase_vmin_vmax[phase] = (vmin, vmax)
    else:
        # fallback if phase missing
        phase_vmin_vmax[phase] = (1e-6, 1)
        


## Plot

In [ ]:
# ============================================================
# --- Plotting (phase maps) & parameters to tune plot --------
# ============================================================

# Number of rows → number of phases
n_rows = len(phases)

# Number of columns → number of samples
n_cols = len(all_data)

# Create subplots
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(1.8 * n_cols + 0.8, 1.8 * n_rows),
    facecolor="black"
)

# Ensure axes is always 2D for consistent indexing
axes = np.atleast_2d(axes)

ims = []

for row_idx, phase in enumerate(phases):
    vmin, vmax = phase_vmin_vmax[phase]
    for col_idx, (sample_label, data) in enumerate(all_data.items()):
        ax = axes[row_idx, col_idx]
        ax.set_facecolor("black")
        ax.axis("off")

        # Plot image
        if phase in data:
            img = data[phase]

            # --- APPLY THRESHOLD TO Nb2O5 tetragonal ---
            if phase == 'Nb2O5 tetragonal':
                threshold = 1.0   # <-- adjust this value
                img = np.where(img >= threshold, img, 0)

            cmap = get_phase_cmap(phase)

            img = np.rot90(img, 2)
            img = np.fliplr(img)

            im = ax.imshow(img, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
        else:
            img_shape = next(iter(data.values())).shape if data else (100, 100)
            black_cmap = LinearSegmentedColormap.from_list("black_only", ["black", "black"])
            im = ax.imshow(np.zeros(img_shape), cmap=black_cmap, vmin=0, vmax=1)
        ims.append(im)

        # Top row: sample titles
        if row_idx == 0:
            ax.set_title(sample_label, fontsize=12, fontname="DejaVu Sans",
                         color="white", fontweight="bold", pad=4)

        # First column: phase labels
        if col_idx == 0:
            phase_label = chem_unicode(phase)
            ax.set_ylabel(phase_label, fontsize=12, fontname="DejaVu Sans",
                          rotation=90, labelpad=10, color="white", fontweight="bold")

# --- Add one colorbar per row (last column) --- #
for row_idx, phase in enumerate(phases):
    ax = axes[row_idx, -1]
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.05)

    sm = plt.cm.ScalarMappable(cmap=get_phase_cmap(phase), norm=plt.Normalize(
        vmin=phase_vmin_vmax[phase][0],
        vmax=phase_vmin_vmax[phase][1]
    ))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)

    # Tick colors
    cbar.ax.tick_params(color='white', labelcolor='white')

    # Label with Unicode subscripts
    phase_label = chem_unicode(phase)
    cbar.set_label(phase_label, color='white', rotation=270,
                    fontsize=10, fontweight='bold', fontname="DejaVu Sans")

    ## Fine-tune label position: shift it further from the bar 
    ## Values: x=15 (further right), y=0.5 (center vertically) cbar.ax.yaxis.set_label_coords(15, 0.5)
    cbar.ax.yaxis.set_label_coords(15, 0.5)


# --- Adjust subplot spacing ---
plt.subplots_adjust(left=0.05, right=0.88, top=0.95, bottom=0.05,
                    wspace=0.05, hspace=0.1)

# wspace → horizontal spacing between 2 columns 
# hspace → vertical spacing between 2 rows 
# left, right, top, bottom → margins around the figure


# --- Scale bar (bottom-left of figure) ---
pixels_per_mm = 320 / 6.0
scale_length_mm = 3.0
scale_length_px = scale_length_mm * pixels_per_mm

ref_ax = axes[-1, 0]
bbox = ref_ax.get_position()
x0_fig = bbox.x0 + 0.0175
y0_fig = bbox.y0 - 0.015
im_width = next(iter(next(iter(all_data.values())).values())).shape[1]
bar_length_frac = (scale_length_px / im_width) * (bbox.x1 - bbox.x0)

fig.lines.append(Line2D([x0_fig, x0_fig + bar_length_frac],
                        [y0_fig, y0_fig],
                        transform=fig.transFigure,
                        color="white",
                        linewidth=3,
                        solid_capstyle="butt"))

fig.text(
    x0_fig + bar_length_frac / 2,
    y0_fig - 0.015,
    f"{scale_length_mm} mm",
    color="white",
    ha="center", va="top",
    fontname="DejaVu Sans",
    fontsize=12,
    fontweight="bold"
)


# --- Save figure at 300 dpi ---
output_path = r"C:\Users\xxx\Desktop\XRD_data\phase_profiles.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="black")
print(f"\n✅ Figure saved at 300 dpi: {output_path}")

plt.show()


### Multi-panel radial profiles

In [ ]:
# ==============================================================================
# Multi-panel radial profiles #
# - One panel per phase
# - Builds FILLED pellet masks directly from e.g. Nb2O5 orthorhombic
# - Radial coordinate based on distance-to-surface
# - core = 0 ; outer region = 1
# - Robust method to investigate the radial profiles of irregular pellet shapes
# - Does NOT require BLOCK 1
# ==============================================================================

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.size"] = 16
mpl.rcParams["axes.titlesize"] = 16
mpl.rcParams["axes.labelsize"] = 16
mpl.rcParams["legend.fontsize"] = 16
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
n_radial_bins = 60
min_pixels_per_bin = 5
use_std_band = False
line_width = 2.2

# Mask-generation settings
# This is the small useful part taken from the old BLOCK 1
mask_reference_phase = "Nb2O5 orthorhombic"
mask_threshold = 0.0
min_valid_pixels = 200

# Optional thresholds for profile cleanup
phase_profile_thresholds = {"Nb2O5 tetragonal": 10.0}
# phase_profile_thresholds = {}

# Filled-mask settings
closing_structure = np.ones((5, 5), dtype=bool)

# Panel sizing
panel_w = 3.5
panel_h = 3.4

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------
if "all_data" not in globals():
    raise ValueError("'all_data' not found.")

if "phases" not in globals():
    raise ValueError("'phases' not found.")

# ------------------------------------------------------------
# Build pellet masks directly here
# ------------------------------------------------------------
sample_masks = {}

sample_order = list(all_data.keys())

for sample in sample_order:

    if mask_reference_phase not in all_data[sample]:
        print(f"Skipping {sample}: phase '{mask_reference_phase}' not found")
        continue

    ref_img = np.array(
        all_data[sample][mask_reference_phase],
        dtype=float,
        copy=True
    )

    # Pellet mask:
    # finite pixels above the chosen threshold define the pellet region
    pellet_mask = np.isfinite(ref_img) & (ref_img > mask_threshold)

    if np.sum(pellet_mask) < min_valid_pixels:
        print(
            f"Skipping {sample}: too few valid pixels in "
            f"'{mask_reference_phase}'"
        )
        continue

    sample_masks[sample] = pellet_mask

if len(sample_masks) == 0:
    raise ValueError("No valid pellet masks were generated.")

print("Pellet masks generated:")
for sample in sample_masks:
    print(f" - {sample}")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def get_phase_image_for_profile(phase, img):
    img_work = np.array(img, dtype=float, copy=True)

    if phase in phase_profile_thresholds:
        thr = phase_profile_thresholds[phase]
        img_work = np.where(img_work >= thr, img_work, 0.0)

    return img_work


def build_filled_mask_from_existing(mask, closing_structure=np.ones((5, 5), dtype=bool)):
    """
    Takes an existing pellet mask and:
    - keeps the largest connected component
    - fills internal holes/cracks
    - applies a mild binary closing to smooth narrow interruptions
    """
    mask = np.asarray(mask, dtype=bool)

    if np.sum(mask) == 0:
        return mask

    lbl, n = label(mask)

    if n > 1:
        sizes = np.bincount(lbl.ravel())
        sizes[0] = 0
        largest_label = np.argmax(sizes)
        mask = lbl == largest_label

    mask = binary_fill_holes(mask)
    mask = binary_closing(mask, structure=closing_structure)

    return mask


def radial_profile_distance_to_edge(img, pellet_mask, n_bins=60, min_pixels_per_bin=5):
    """
    Radial coordinate based on distance-to-surface:
      - outer region / surface = 1
      - core = 0

    For each pixel inside the pellet mask:
      1) compute Euclidean distance to the nearest boundary
      2) normalize by the maximum internal distance
      3) invert so that:
            r/R = 0  -> core
            r/R = 1  -> outer region
    """
    pellet_mask = np.asarray(pellet_mask, dtype=bool)
    img = np.asarray(img, dtype=float)

    if img.shape != pellet_mask.shape:
        return None, None, None

    if np.sum(pellet_mask) == 0:
        return None, None, None

    # Distance to nearest pixel outside the mask
    dist_map = distance_transform_edt(pellet_mask)

    dist_valid = dist_map[pellet_mask]
    val_valid = img[pellet_mask]

    if dist_valid.size == 0:
        return None, None, None

    d_max = dist_valid.max()

    if d_max == 0:
        return None, None, None

    # Invert normalized distance:
    # core -> 0 ; surface -> 1
    rr_norm = 1.0 - (dist_valid / d_max)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    inds = np.digitize(rr_norm, bins) - 1
    inds = np.clip(inds, 0, n_bins - 1)

    prof_mean = np.full(n_bins, np.nan)
    prof_std = np.full(n_bins, np.nan)

    for i in range(n_bins):
        m = inds == i

        if np.sum(m) >= min_pixels_per_bin:
            prof_mean[i] = np.nanmean(val_valid[m])
            prof_std[i] = np.nanstd(val_valid[m])

    r_centers = 0.5 * (bins[:-1] + bins[1:])

    return r_centers, prof_mean, prof_std


def infer_stage_rank(sample_name):
    """
    Lower rank = earlier stage, higher rank = later stage.
    This controls color darkness: later = darker.
    """
    s = sample_name.lower().strip()

    if "calcined" in s:
        return 0

    if "reduced" in s:
        return 1

    import re
    m = re.search(r'(\d+)\s*h', s)

    if m:
        return 10 + int(m.group(1))

    return 5


# -----------------------------------------------------------------------------
# Determine sample order and colors: swap the color map "inferno" if preferred
# -----------------------------------------------------------------------------
sample_list = [s for s in all_data.keys() if s in sample_masks]
sample_list = sorted(sample_list, key=infer_stage_rank)

if len(sample_list) == 0:
    raise ValueError("No samples available after mask generation.")

cmap = plt.get_cmap("inferno")

if len(sample_list) == 1:
    color_positions = [0.55]
else:
    color_positions = np.linspace(0.85, 0.15, len(sample_list))

sample_colors = {
    sample: cmap(pos)
    for sample, pos in zip(sample_list, color_positions)
}

# ------------------------------------------------------------
# Keep only phases that have at least one usable sample
# ------------------------------------------------------------
usable_phases = []

for phase in phases:
    found = False

    for sample_label in sample_list:
        if phase in all_data.get(sample_label, {}):
            found = True
            break

    if found:
        usable_phases.append(phase)

if len(usable_phases) == 0:
    raise ValueError("No usable phases found for plotting.")

# ------------------------------------------------------------
# Multi-panel layout
# ------------------------------------------------------------
n_phases = len(usable_phases)
ncols = min(3, n_phases)
nrows = math.ceil(n_phases / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(panel_w * ncols, panel_h * nrows),
    squeeze=False
)

axes_flat = axes.flatten()

radial_profiles = {}

# ------------------------------------------------------------
# Plot each phase in one panel
# ------------------------------------------------------------
for ax, phase in zip(axes_flat, usable_phases):

    radial_profiles[phase] = {}
    any_curve = False

    for sample_label in sample_list:

        if phase not in all_data.get(sample_label, {}):
            continue

        img = np.array(
            all_data[sample_label][phase],
            dtype=float,
            copy=True
        )

        img = get_phase_image_for_profile(phase, img)

        pellet_mask_raw = sample_masks[sample_label]

        pellet_mask = build_filled_mask_from_existing(
            pellet_mask_raw,
            closing_structure=closing_structure
        )

        if img.shape != pellet_mask.shape:
            print(
                f"Skipping {phase} / {sample_label}: "
                f"shape mismatch {img.shape} vs {pellet_mask.shape}"
            )
            continue

        result = radial_profile_distance_to_edge(
            img=img,
            pellet_mask=pellet_mask,
            n_bins=n_radial_bins,
            min_pixels_per_bin=min_pixels_per_bin
        )

        if result[0] is None:
            continue

        r, mean, std = result

        radial_profiles[phase][sample_label] = {
            "r": r,
            "mean": mean,
            "std": std,
        }

        color = sample_colors[sample_label]

        valid = np.isfinite(mean)

        if np.sum(valid) < 2:
            continue

        r_plot = r[valid]
        mean_plot = mean[valid]
        std_plot = std[valid]

        ax.plot(
            r_plot,
            mean_plot,
            color=color,
            linewidth=line_width,
            label=sample_label
        )

        if use_std_band:
            ax.fill_between(
                r_plot,
                mean_plot - std_plot,
                mean_plot + std_plot,
                color=color,
                alpha=0.14
            )

        any_curve = True

    title_phase = chem_unicode(phase) if "chem_unicode" in globals() else phase

    ax.set_title(title_phase, fontweight="bold")
    ax.set_xlabel("r/R (a.u.)")
    ax.set_ylabel("wt.%")
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.22)

    if not any_curve:
        ax.text(
            0.5,
            0.5,
            "No data",
            ha="center",
            va="center",
            transform=ax.transAxes
        )

# Hide unused axes
for j in range(len(usable_phases), len(axes_flat)):
    axes_flat[j].axis("off")

# ------------------------------------------------------------
# One common legend for the whole figure
# ------------------------------------------------------------
legend_handles = [
    Line2D(
        [0],
        [0],
        color=sample_colors[s],
        lw=line_width,
        label=s
    )
    for s in sample_list
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.04)
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\nRadial profile summary:")
print("Samples used, early -> late:")

for s in sample_list:
    print(f" - {s}")

print("\nPhases plotted:")

for phase, d in radial_profiles.items():
    if len(d) == 0:
        print(f" - {phase}: no usable curves")
    else:
        print(f" - {phase}: {list(d.keys())}")

# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------
output_path = (r"C:\Users\xxx\Desktop\XRD_data\sample_profiles_filled_mask.png")

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"\n✅ Figure saved at 300 dpi: {output_path}")

plt.show()




### Optional diagnostic plot: iso-contours and radial-bin histogram

In [ ]:
# ============================================================
# Iso-contours + histogram + radial profile
# Choose one sample and one phase to show the iso-contours
# ============================================================

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["legend.fontsize"] = 12
mpl.rcParams["xtick.labelsize"] = 12
mpl.rcParams["ytick.labelsize"] = 12
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------
sample_to_plot = "25 h"
phase_to_plot = "Ru"
mask_phase = "Nb2O5 orthorhombic"

n_radial_bins = 60
min_pixels_per_bin = 5

# Optional cleanup threshold for displayed/profiled phase
phase_thresholds = {
    # "Nb2O5 tetragonal": 5.0,
    # "Ru": 0.2,
}

# Plot limits
cbar_min = 0
cbar_max = 10

profile_ymin = 0
profile_ymax = 2.5

# Contour styling
contour_color = "white"
boundary_color = "yellowgreen"
contour_lw = 1.3
boundary_lw = 1.8
n_iso = 6


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------
if "all_data" not in globals():
    raise ValueError("Run the block that defines 'all_data' first.")

if sample_to_plot not in all_data:
    raise ValueError(f"Sample '{sample_to_plot}' not found in all_data.")

if phase_to_plot not in all_data[sample_to_plot]:
    raise ValueError(f"Phase '{phase_to_plot}' not found for sample '{sample_to_plot}'.")

if mask_phase not in all_data[sample_to_plot]:
    raise ValueError(f"Mask phase '{mask_phase}' not found for sample '{sample_to_plot}'.")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def chem_unicode(name):
    subscript_map = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    return "".join(c.translate(subscript_map) if c.isdigit() else c for c in name)


def apply_phase_threshold(phase, img):
    img = np.array(img, dtype=float, copy=True)

    if phase in phase_thresholds:
        threshold = phase_thresholds[phase]
        img = np.where(img >= threshold, img, 0.0)

    return img


def build_filled_pellet_mask(mask_img):
    """
    Builds pellet mask from a phase map:
    - finite values > 0 are treated as pellet
    - largest connected component retained
    - holes/cracks filled
    - slight closing applied
    """
    mask = np.isfinite(mask_img) & (mask_img > 0)

    if np.sum(mask) == 0:
        raise ValueError("Initial mask is empty.")

    lbl, n = label(mask)

    if n > 1:
        sizes = np.bincount(lbl.ravel())
        sizes[0] = 0
        mask = lbl == np.argmax(sizes)

    mask = binary_fill_holes(mask)
    mask = binary_closing(mask, structure=np.ones((5, 5), dtype=bool))

    return mask


def radial_profile_distance_to_edge(img, pellet_mask, n_bins=60, min_pixels_per_bin=5):
    """
    Distance-to-surface radial profile:
    core = 0
    outer region = 1
    """
    pellet_mask = np.asarray(pellet_mask, dtype=bool)
    img = np.asarray(img, dtype=float)

    if img.shape != pellet_mask.shape:
        raise ValueError(f"Shape mismatch: img {img.shape} vs mask {pellet_mask.shape}")

    if np.sum(pellet_mask) == 0:
        raise ValueError("Pellet mask is empty.")

    # Distance from each pellet pixel to the nearest external pixel
    dist_map = distance_transform_edt(pellet_mask)

    dist_valid = dist_map[pellet_mask]
    val_valid = img[pellet_mask]

    d_max = dist_valid.max()

    if d_max == 0:
        raise ValueError("Maximum internal distance is zero.")

    # Normalized radial coordinate:
    # core -> 0
    # surface -> 1
    rr_valid = 1.0 - (dist_valid / d_max)

    # Map version for iso-contours
    rr_map = np.full(img.shape, np.nan, dtype=float)
    rr_map[pellet_mask] = rr_valid

    # Binning
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    inds = np.digitize(rr_valid, bins) - 1
    inds = np.clip(inds, 0, n_bins - 1)

    prof_mean = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)

    for i in range(n_bins):
        m = inds == i
        counts[i] = np.sum(m)

        if counts[i] >= min_pixels_per_bin:
            prof_mean[i] = np.nanmean(val_valid[m])

    r_centers = 0.5 * (bins[:-1] + bins[1:])

    return r_centers, prof_mean, counts, rr_map, bins


# ------------------------------------------------------------
# Load chosen sample/phase and build mask
# ------------------------------------------------------------
img_phase = np.array(all_data[sample_to_plot][phase_to_plot], dtype=float, copy=True)
img_phase = apply_phase_threshold(phase_to_plot, img_phase)

img_mask_phase = np.array(all_data[sample_to_plot][mask_phase], dtype=float, copy=True)
pellet_mask = build_filled_pellet_mask(img_mask_phase)

# ------------------------------------------------------------
# Compute radial quantities
# ------------------------------------------------------------
r, prof_mean, counts, rr_map, bins = radial_profile_distance_to_edge(
    img=img_phase,
    pellet_mask=pellet_mask,
    n_bins=n_radial_bins,
    min_pixels_per_bin=min_pixels_per_bin
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig = plt.figure(figsize=(13.5, 4.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.9, 1.0])

title_phase = chem_unicode(phase_to_plot)

# ------------------------------------------------------------
# (a) Phase map + iso-contours + mask boundary
# ------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

im = ax1.imshow(
    img_phase,
    cmap="inferno",
    vmin=cbar_min,
    vmax=cbar_max
)

contour_levels = np.linspace(0.1, 0.9, n_iso)

ax1.contour(
    rr_map,
    levels=contour_levels,
    colors=contour_color,
    linewidths=contour_lw
)

ax1.contour(
    pellet_mask.astype(float),
    levels=[0.5],
    colors=boundary_color,
    linewidths=boundary_lw
)

ax1.set_title(f"a) {title_phase} map")
ax1.set_axis_off()

cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.set_label("wt.%")

# ------------------------------------------------------------
# (b) Histogram of pixels per radial bin
# ------------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])

centers = 0.5 * (bins[:-1] + bins[1:])
widths = np.diff(bins)

bars = ax2.bar(
    centers,
    counts,
    width=widths * 0.92,
    edgecolor="black",
    linewidth=0.8,
    color=plt.get_cmap("Oranges")(centers)
)

for i, bar in enumerate(bars):
    if counts[i] < min_pixels_per_bin:
        bar.set_hatch("//")

ax2.set_title("b) Radial binning histogram")
ax2.set_xlabel("r/R (a.u.)")
ax2.set_ylabel("Pixel count")
ax2.set_xlim(0, 1)
ax2.grid(True, alpha=0.22)

# ------------------------------------------------------------
# (c) Radial profile
# ------------------------------------------------------------
ax3 = fig.add_subplot(gs[0, 2])

valid = np.isfinite(prof_mean)

ax3.plot(
    r[valid],
    prof_mean[valid],
    color="black",
    linewidth=2.2
)

ax3.set_title(f"c) Radial profile of {title_phase}")
ax3.set_xlabel("r/R (a.u.)")
ax3.set_ylabel("wt.%")
ax3.set_xlim(0, 1)
ax3.set_ylim(profile_ymin, profile_ymax)
ax3.grid(True, alpha=0.22)

plt.tight_layout()

# ------------------------------------------------------------
# Save if desired
# ------------------------------------------------------------
# Optional save
save_figure = False
save_path = r"C:\Users\xxx\Desktop\XRD_data\radial_schematic.png"

if save_figure:
    fig.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"✅ Figure saved at 300 dpi: {save_path}")

plt.show()

# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------
print("\nSummary")
print(f"Sample plotted       : {sample_to_plot}")
print(f"Profile phase        : {phase_to_plot}")
print(f"Mask phase           : {mask_phase}")
print(f"Number of radial bins: {n_radial_bins}")
print(f"Minimum pixels/bin   : {min_pixels_per_bin}")
print(f"Valid profile bins   : {np.sum(np.isfinite(prof_mean))} / {len(prof_mean)}")